In [2]:
# Fix path
import sys
import os
sys.path.append(os.path.abspath(".."))

# Now imports
from src.data_loader import load_data
from src.advanced_model import train_xgboost, train_lightgbm

import pandas as pd

In [3]:
df = load_data()
df.columns = df.columns.str.lower()

print(df.shape)
df.head()

(255347, 18)


,loanid,age,income,loanamount,creditscore,monthsemployed,numcreditlines,interestrate,loanterm,dtiratio,education,employmenttype,maritalstatus,hasmortgage,hasdependents,loanpurpose,hascosigner,default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [4]:
# Create X and y
X = df.drop('default', axis=1)
y = df['default']

# Keep numeric only (important)
X = X.select_dtypes(include=['int64','float64'])

print(X.shape)

(255347, 9)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
xgb_model = train_xgboost(X_train, X_test, y_train, y_test)

XGBoost Accuracy: 0.887546504797337
XGBoost AUC: 0.7527884620435792


In [7]:
lgb_model = train_lightgbm(X_train, X_test, y_train, y_test)

[LightGBM] [Info] Number of positive: 23753, number of negative: 180524
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013833 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1285
[LightGBM] [Info] Number of data points in the train set: 204277, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116278 -> initscore=-2.028155
[LightGBM] [Info] Start training from score -2.028155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [8]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'max_depth': [3, 5],
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1]
}

grid = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    scoring='roc_auc',
    cv=3
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best AUC:", grid.best_score_)

Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best AUC: 0.742264950555803


In [9]:
best_model = grid.best_estimator_

from sklearn.metrics import roc_auc_score

y_prob = best_model.predict_proba(X_test)[:,1]

print("Tuned XGBoost AUC:", roc_auc_score(y_test, y_prob))

Tuned XGBoost AUC: 0.7537997921224151


In [10]:
from sklearn.metrics import roc_auc_score

print("Baseline AUC: 0.74")

print("XGBoost AUC:",
      roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]))

print("LightGBM AUC:",
      roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:,1]))

print("Tuned XGBoost AUC:",
      roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1]))

Baseline AUC: 0.74
XGBoost AUC: 0.7527884620435792
LightGBM AUC: 0.7521310116584053
Tuned XGBoost AUC: 0.7537997921224151


In [11]:
import pickle

with open("../model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Model saved successfully!")

Model saved successfully!


In [12]:
X.shape

(255347, 9)

In [13]:
df.drop('default', axis=1).select_dtypes(include=['int64','float64']).columns

Index(['age', 'income', 'loanamount', 'creditscore', 'monthsemployed',
       'numcreditlines', 'interestrate', 'loanterm', 'dtiratio'],
      dtype='object')